### Which products have the highest revenue?

In [0]:
SELECT
product_id,
product_name,
SUM(revenue) AS product_revenue,
SUM(quantity) AS units_sold
FROM shopflow_analytics_cat.gold.fct_order_items
GROUP BY product_id,product_name
ORDER BY product_revenue DESC
LIMIT 20;

product_id,product_name,product_revenue,units_sold
prod_001,Wireless Headphones,549.89,11
prod_007,Monitor Stand,513.04,4
prod_002,Bluetooth Speaker,491.00,5
prod_004,Mechanical Keyboard,436.37,5
prod_006,Notebook Set,299.50,4
prod_005,Desk Lamp,285.48,4
prod_003,USB-C Hub,284.01,4
prod_008,Desk Organizer,81.72,2


### Which categories drive the most revenue?

In [0]:
SELECT
    product_category AS category,
    SUM(revenue)     AS category_revenue,
    SUM(quantity)    AS units_sold
FROM shopflow_analytics_cat.gold.fct_order_items
GROUP BY product_category
ORDER BY category_revenue DESC;


category,category_revenue,units_sold
Electronics,2274.31,29
Office,381.22,6
Home,285.48,4


###What is average order value (AOV) by month?

In [0]:
SELECT
    DATE_TRUNC('month', order_date)        AS month,
    SUM(revenue)                           AS total_revenue,
    COUNT(DISTINCT order_id)               AS order_count,
    SUM(revenue) / COUNT(DISTINCT order_id) AS aov
FROM shopflow_analytics_cat.gold.fct_orders
GROUP BY 1
ORDER BY 1;


month,total_revenue,order_count,aov
2024-06-01T00:00:00.000Z,1756.72,12,146.393333333333333333
2024-07-01T00:00:00.000Z,1104.25,8,138.031250000000000000


###  Which countries generate the most revenue?

In [0]:
SELECT
    customer_country            AS country,
    SUM(revenue)                AS revenue,
    COUNT(DISTINCT order_id)    AS order_count
FROM shopflow_analytics_cat.gold.fct_orders
GROUP BY customer_country
ORDER BY revenue DESC
LIMIT 10;


country,revenue,order_count
US,1892.48,11
UK,555.50,6
CA,278.00,2
DE,134.99,1


### Who are the top customers by revenue?

In [0]:
SELECT
    o.customer_id,
    MAX(c.email)        AS email,
    MAX(c.country)      AS country,
    SUM(o.revenue)      AS total_revenue,
    COUNT(o.order_id)   AS order_count
FROM shopflow_analytics_cat.gold.fct_orders o
LEFT JOIN shopflow_analytics_cat.gold.scd_customers c
    ON o.customer_id = c.customer_id
   AND c.dbt_valid_to IS NULL
GROUP BY o.customer_id
ORDER BY total_revenue DESC
LIMIT 20;


customer_id,email,country,total_revenue,order_count
cust_001,alice.j_updated@email.com,US,671.99,4
cust_005,eve.davis@email.com,US,490.75,2
cust_004,david.b@email.com,CA,278.00,2
cust_003,carol.w@email.com,US,268.99,2
cust_009,ivy.t@email.com,US,267.00,1
cust_002,bob.smith@email.com,UK,257.50,3
cust_006,frank.m@email.com,UK,243.50,2
cust_007,grace.w@email.com,US,193.75,2
cust_008,henry.moore@email.com,DE,134.99,1
cust_010,jack.a@email.com,UK,54.50,1


### Who are the customers who have more than one order (repeat customers)?

In [0]:
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.no_of_orders
FROM (
    SELECT
        customer_id,
        COUNT(order_id) as no_of_orders
    FROM shopflow_analytics_cat.gold.fct_orders
    WHERE customer_id IS NOT NULL
      AND order_id IS NOT NULL
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
) o
JOIN shopflow_analytics_cat.gold.scd_customers c
    ON o.customer_id = c.customer_id;


customer_id,first_name,last_name,no_of_orders
cust_002,Bob,Smith,3
cust_003,Carol,Williams,2
cust_004,David,Brown,2
cust_005,Eve,Davis,2
cust_006,Frank,Miller,2
cust_007,Grace,Wilson,2
cust_001,Alice,Johnson,4
cust_001,Alice,Johnson,4
